# Basics of Sound
* Speech is changing air pressure over time. (basically a profile of 'pressure' vs 'time').
* digital audio signals are long list of numbers.
* Speech --> Microphone (pressure sensor) --> electrical signals --> numbers 
* Sound in nature is continous, but we sample it and store it at some sample rate.
* Clean for background noise/hiss, remove silence at start/end.
* Most speech model expects: 16khz mono.
* human speech roughly contains usefule frequencies upto 8000Hz, Nyquist criterion requires us to sample at 16000Hz.
* MP3 --> lOSSY, WAV --> Lossless
* mono: two microphones L and R :we average them. --> half storage needed.
* Normalize--> very loud, very quiet :: scales the waveform so that we have a more consistent loudness.



# Whisper ASR Input Requirements & Preprocessing

> **Design Principle:** The purpose of preprocessing is **not** to make audio sound better to humans. Its purpose is to make the audio match the format Whisper's model architecture requires, and — where evidence supports it — to improve the signal-to-noise ratio without removing speech information.

> **Methodology note:** Each row below is tagged with its evidence source. Tier 1 claims are confirmed directly against OpenAI's own `whisper/audio.py` source code and official model cards. Tier 2 claims are widely-reported empirical/community findings, not official spec. Tier 3 claims are general audio-engineering practices with no Whisper-specific confirmation found — treat as untested hypotheses for your own pipeline, not settled requirements.

---

## Tier 1 — Confirmed, hardcoded in OpenAI's source (highest confidence)

| Requirement | Why the model wants this | Preprocessing | Failure if Ignored | Source |
|---|---|---|---|---|
| **16 kHz sample rate** | `SAMPLE_RATE = 16000` is a hardcoded constant in Whisper's feature extraction | Resample to 16 kHz | Feature extraction mismatch, degraded accuracy | OpenAI `whisper/audio.py` |
| **Mono waveform** | `load_audio()` decodes via ffmpeg directly to mono | Convert stereo → mono (unless channels are separate speakers you want to keep distinct) | Channel imbalance, wasted compute | OpenAI `whisper/audio.py` |
| **Automatic decoding of compressed audio** | Whisper's own `load_audio()` shells out to ffmpeg internally | None needed manually for common formats (mp3, m4a, etc.) — Whisper handles this itself | N/A for standard formats | OpenAI `whisper/audio.py` |
| **Float32 normalized PCM** | Model computes in FP32 by default; `load_audio()` returns float32 | Ensure float32 output if bypassing Whisper's own loader | Framework/dtype errors | OpenAI `whisper/audio.py`, model card |
| **30-second chunking** | `CHUNK_LENGTH = 30`, `N_SAMPLES = 480000` — fixed architectural window | Split long audio into ~30s segments; short audio is zero-padded automatically | Truncation, context loss, wasted compute on padding | OpenAI `whisper/audio.py` |
| **Log-mel spectrogram: 80 bins (≤v2) / 128 bins (v3+)** | Feature extraction front-end, version-dependent | N/A — handled internally by Whisper's pipeline | N/A | OpenAI `whisper-large-v3` model card |
| **No native streaming** | Encoder-decoder processes the *full* 30s window before emitting any tokens | Chunk-boundary stitching needed for **long-form audio**; naive concatenation can duplicate/garble words at edges | Garbled transcript at chunk boundaries | Architecture docs, `faster-whisper` known-issues discussion |

---

## Tier 2 — Real, widely-reported empirical findings (not official spec, but well-documented community consensus)

| Requirement | Why it matters | Preprocessing | Failure if Ignored | Confidence |
|---|---|---|---|---|
| **Hallucinations on silent/music-only regions** | Widely reported failure mode in community discussions and papers | Voice Activity Detection (VAD) to skip non-speech regions | Model invents plausible-sounding but fabricated text during silence | ⭐⭐⭐⭐ (empirical, not OpenAI-documented) |
| **Conditioning on previous chunk's text is a hallucination source** | Long-form decoding optionally conditions on prior output; errors can compound | Disable previous-text conditioning, or use independent chunk decoding | Error propagation across chunks | ⭐⭐⭐⭐ |
| **Good signal-to-noise ratio helps** | General ASR principle, consistently reported | Noise suppression *only when noise is genuinely significant* | Word substitutions | ⭐⭐⭐⭐ |
| **Trimming long silence saves compute** | Reduces wasted processing on 30s-padded silent chunks | Trim leading/trailing silence | Increased inference time only (not accuracy) | ⭐⭐⭐ |

---

## Tier 3 — Generic audio-engineering practices (no Whisper-specific source found — test before trusting)

These are legitimate general audio-processing techniques, but **no Whisper-specific documentation or benchmark was found confirming they improve Whisper's accuracy**. Treat as hypotheses to test on your own audio, not defaults to apply blindly.

- Dereverberation
- Declipping / clipping repair
- DC offset removal
- LUFS loudness normalization (vs. simple peak normalization)
- Dynamic range compression
- Music suppression / source separation
- Speaker diarization
- Aggressive stereo channel balancing

> **Why this tier matters for your project:** these are exactly the kind of "sounds rigorous, unverified for this specific model" additions that can cost time without proven benefit — apply the same discipline used elsewhere in this project: test one at a time, on your actual audio file, and keep only what measurably improves transcription accuracy.

---

## Practical Build Order (Tier 1 only, then test)

1. Decode → mono → 16kHz → float32 (all handled automatically if you use Whisper's own `load_audio()`)
2. Chunk into 30s segments if audio is long
3. Run baseline transcription — no Tier 2/3 preprocessing yet
4. **Only if baseline transcript is poor**, apply Tier 2 fixes one at a time (VAD first, since it has the strongest evidence), re-test after each
5. Reserve Tier 3 techniques for stubborn cases, and validate each empirically against your specific "buzzing" audio before trusting it

## Always Do (revised)
- Use Whisper's own `load_audio()` / equivalent standard decode path (handles format, mono, 16kHz, float32 automatically)
- Chunk appropriately for long audio
- Test baseline before adding preprocessing

## Test Empirically Before Trusting
- VAD (strong evidence, but test on your file)
- Noise suppression (only if genuinely noisy)
- Any Tier 3 technique

## Avoid Applying by Default
- Any Tier 3 technique without a measured before/after comparison
- Heavy denoising, aggressive filtering, time stretching — all carry real risk of removing speech information along with noise

### **preprocessing is simply the process of making your audio look more like the audio the model was trained on.**
### **Every preprocessing step should make the input audio distribution look closer to the audio distribution the model saw during training, while avoiding operations that remove speech information.**

* Can I read this file?
*    ↓
* Decode

* Does the model expect this format?
*    ↓
* Mono
* Resample
* Bit depth

* Can the model hear the speech?
*    ↓
* Noise reduction
* Dereverberation
* Normalization

* Is there unnecessary audio?
*     ↓
* Trim silence
* VAD
* Chunking

* Is the recording fundamentally damaged?
*    ↓
* Clipping detection
* Quality checks


                MP3 / WAV / FLAC
                        │
                        ▼
                Decode (FFmpeg)
                        │
                        ▼
              Convert to Mono
                        │
                        ▼
             Resample to 16 kHz
                        │
                        ▼
      Convert PCM → float32 [-1,1]
                        │
                        ▼
         Split into ~30 s windows
                        │
                        ▼
          Compute Log-Mel Spectrogram
                        │
                        ▼
        Transformer Encoder (acoustics)
                        │
                        ▼
          Automatic Language Detection
                        │
                        ▼
      Transformer Decoder + Beam Search
                        │
                        ▼
         Timestamped Text Segments
                        │
                        ▼
         Join Segments into Transcript

In [1]:
from faster_whisper import WhisperModel

def transcribe_audio(file_path: str, model_size: str = "base") -> str:
    """Tier 1 baseline — no manual preprocessing. Library handles
    decode -> mono -> 16kHz -> float32 -> 30s chunking internally."""
    model = WhisperModel(model_size, device="cpu", compute_type="int8")
    segments, info = model.transcribe(file_path, beam_size=5)
    transcript = " ".join(seg.text.strip() for seg in segments)
    return transcript

/Users/krahuldnkr/Final_Assignment_Template/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
transcript = transcribe_audio("artifacts/99c9cc74-fdc8-46c6-8f8d-3ce2d3bfeea3.mp3")
print(transcript)

In a saucepan, combine ripe strawberries, granulated sugar, freshly squeezed lemon juice and cornstarch. Cook the mixture over medium heat, stirring constantly until it thickens to a smooth consistency. Remove from heat and stir in a dash of pure vanilla extract. Allow the strawberry pie feeling to cool before using it as a delicious and fruity feeling for your pie crust.


* Output compared to Ground Truth::
"In a saucepan, combine ripe strawberries, granulated sugar, 
freshly squeezed lemon juice and cornstarch. 
Cook the mixture over medium heat, stirring constantly 
until it thickens to a smooth consistency. 
Remove from heat and stir in a dash of pure vanilla extract. 
Allow the strawberry pie feeling to cool
before using it as a delicious and fruity feeling for your pie crust."

In [4]:
transcript = transcribe_audio("artifacts/1f975693-876d-457b-a649-393859e79bf3.mp3")
print(transcript)

Before you all go, I want to remind you that the midterm is next week. Here's a little hint. You should be familiar with the differential equations on page 245. Problems that are very similar to problems 32, 33 and 44 from that page might be on the test. And also some of you might want to brush up on the last page in the integration section, page 197. I know some of you struggled on last week's quiz. I foresee problem 22 from page 197 being on your midterm. Oh and don't forget to brush up on the section on related rates on pages 132, 133 and 134.


* Output 2 (Comparison with ground truth): 
Before you all go, I want to remind you that the midterm
is next week. Here's a little hint. You should be familiar 
with the differential equations on page 245. Problems that 
are very similar to problems 32, 33 and 44 from that page
might be on the test. And also some of you might want to 
brush up on the last page in the integration section, page
197. I know some of you struggled on last week's quiz. I '
'foresee problem 22 from page 197 being on your midterm.'
' Oh and don't forget to brush up on the section on related 
rates on pages 132, 133 and 134.